# Лабораторна 5. Завдання 4 (частина 1) — CNN_MNIST: навчання, порівняння архітектур, k-fold, збереження
Пункти 12–33.

## 12. Імпорт

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import KFold

tf.keras.utils.set_random_seed(42)

## 13–14. Дані: [зразки][ширина][висота][канали=1], нормалізація, one-hot

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1).astype("float32") / 255
X_test  = X_test.reshape(-1, 28, 28, 1).astype("float32") / 255
y_train_cat = to_categorical(y_train)
y_test_cat  = to_categorical(y_test)
num_classes = y_test_cat.shape[1]
print(X_train.shape, y_train_cat.shape)

In [ ]:
def plot_history(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h.history["accuracy"], label="train"); ax[0].plot(h.history["val_accuracy"], label="val")
    ax[0].set_title(title + " — accuracy"); ax[0].set_xlabel("епоха"); ax[0].legend()
    ax[1].plot(h.history["loss"], label="train"); ax[1].plot(h.history["val_loss"], label="val")
    ax[1].set_title(title + " — loss"); ax[1].set_xlabel("епоха"); ax[1].legend()
    plt.show()

results = {}

## 15–24. Проста CNN: Conv(32, 5×5) → MaxPool 2×2 → Dropout 0.2 → Flatten → Dense 128 → Dense 10

In [ ]:
def cnn_simple():
    m = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (5, 5), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ])
    m.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return m

m1 = cnn_simple(); m1.summary()
h1 = m1.fit(X_train, y_train_cat, validation_data=(X_test, y_test_cat), epochs=10, batch_size=200, verbose=2)
s = m1.evaluate(X_test, y_test_cat, verbose=0)
results["CNN simple"] = s[1]
print("Помилка CNN simple: %.2f%%" % (100 - s[1] * 100))
plot_history(h1, "CNN simple")

## 25–26. Складніша CNN: Conv30(5×5) → Pool → Conv15(3×3) → Pool → Dropout 0.2 → Flatten → 128 → 50 → 10

In [ ]:
def cnn_medium():
    m = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(30, (5, 5), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(15, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(50, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ])
    m.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return m

m2 = cnn_medium(); m2.summary()
h2 = m2.fit(X_train, y_train_cat, validation_data=(X_test, y_test_cat), epochs=10, batch_size=200, verbose=2)
s = m2.evaluate(X_test, y_test_cat, verbose=0)
results["CNN medium"] = s[1]
print("Помилка CNN medium: %.2f%%" % (100 - s[1] * 100))
plot_history(h2, "CNN medium")

## 27–28. k-fold крос-валідація (k = 5) для ускладненої мережі

In [ ]:
def cnn_complex():
    # ускладнена мережа: подвійні згортки + BatchNorm + Dropout
    m = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(64, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ])
    m.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return m

FAST = True   # True — швидкий режим (3 епохи на fold); False — повний (8 епох)

def kfold_eval(build_fn, k=5, epochs=None, batch_size=256):
    epochs = epochs or (3 if FAST else 8)
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    accs, histories = [], []
    for fold, (tr, va) in enumerate(kf.split(X_train), 1):
        model = build_fn()
        es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)
        h = model.fit(X_train[tr], y_train_cat[tr], validation_data=(X_train[va], y_train_cat[va]),
                      epochs=epochs, batch_size=batch_size, verbose=0, callbacks=[es])
        acc = model.evaluate(X_train[va], y_train_cat[va], verbose=0)[1]
        accs.append(acc); histories.append(h)
        print(f"Fold {fold}: accuracy = {acc:.4f}")
    print(f"Середня точність: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    return accs, histories

accs_complex, hist_complex = kfold_eval(cnn_complex)
plt.boxplot(accs_complex); plt.title("5-fold: accuracy ускладненої CNN"); plt.show()

## 29. Ще глибша мережа

In [ ]:
def cnn_deep():
    m = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(64, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ])
    m.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return m

accs_deep, hist_deep = kfold_eval(cnn_deep)

## 30–31. Вибір остаточної моделі
Порівняйте середні значення k-fold: найкращою (зазвичай `cnn_deep`) обираємо як остаточну.

In [ ]:
print("Тест на одному поділі:", {k: f"{v:.4f}" for k, v in results.items()})
print("k-fold complex: %.4f | k-fold deep: %.4f" % (np.mean(accs_complex), np.mean(accs_deep)))
final_builder = cnn_deep if np.mean(accs_deep) >= np.mean(accs_complex) else cnn_complex
print("Остаточна модель:", final_builder.__name__)

## 32–33. Фінальне навчання на всіх даних, захист від перенавчання, криві навчання, збереження

In [ ]:
final = final_builder()
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("cnn_mnist_best.keras", monitor="val_accuracy", save_best_only=True),
]
# validation_split — частина навчальних даних для контролю перенавчання (тест лишається незалежним)
hf = final.fit(X_train, y_train_cat, validation_split=0.1, epochs=12 if FAST else 30, batch_size=256,
               verbose=2, callbacks=callbacks)
plot_history(hf, "Фінальна модель")

test_loss, test_acc = final.evaluate(X_test, y_test_cat, verbose=0)
print(f"Тест: accuracy = {test_acc:.4f}, помилка = {100 - test_acc * 100:.2f}%")
final.save("cnn_mnist_final.keras")

### Оцінка збереженої моделі на 10 зображеннях датасету

In [ ]:
loaded = keras.models.load_model("cnn_mnist_final.keras")
idx = list(range(10))
pred = loaded.predict(X_test[idx], verbose=0).argmax(axis=1)
fig, axes = plt.subplots(1, 10, figsize=(16, 2.2))
for a, i, p in zip(axes, idx, pred):
    a.imshow(X_test[i].reshape(28, 28), cmap="gray"); a.axis("off")
    a.set_title(f"{p} ({'ok' if p == y_test[i] else 'ПОМИЛКА'})", fontsize=9)
plt.savefig("cnn_dataset_predictions.png", dpi=150, bbox_inches="tight"); plt.show()
print("Правильно:", (pred == y_test[idx]).sum(), "із 10")